In [6]:
import pandas as pd
import numpy as np
import sqlalchemy as sa
from sqlalchemy.engine import URL

from sklearn.metrics import mean_absolute_error, mean_squared_error

from statsmodels.tsa.holtwinters import SimpleExpSmoothing
from statsmodels.tsa.statespace.sarimax import SARIMAX

from prophet import Prophet  # pip install prophet


In [7]:
# ---------- 1. Connect to SQL Server ----------

connection_string = (
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=localhost;"      # change as needed
    "DATABASE=EMSData;"      # change as needed
    "Trusted_Connection=yes;"
)

connection_url = URL.create(
    "mssql+pyodbc",
    query={"odbc_connect": connection_string}
)

engine = sa.create_engine(connection_url)


In [8]:
# ---------- 2. Read raw data in chunks and build daily series ----------

chunksize = 100000
sql = """
SELECT PcrKey, EventTime_raw, exposure_flag
FROM dbo.PCR_Exposure_Minimal
"""

daily_counts = {}
rows_processed = 0

chunks = pd.read_sql(sql, engine, chunksize=chunksize)

for i, chunk in enumerate(chunks, start=1):
    rows_processed += len(chunk)

    # parse datetime
    chunk["event_dt"] = pd.to_datetime(
        chunk["EventTime_raw"].astype(str).str.strip(),
        format="%d%b%Y:%H:%M:%S",
        errors="coerce",
    )

    # keep valid datetime and exposure_flag == 1
    chunk = chunk[
        chunk["event_dt"].notna() &
        (chunk["exposure_flag"] == 1)
    ]

    # count per calendar day
    vc = chunk["event_dt"].dt.date.value_counts()
    for d, c in vc.items():
        daily_counts[d] = daily_counts.get(d, 0) + c

    # progress every ~500k rows
    if rows_processed % 500000 < chunksize:
        print(f"Processed about {rows_processed:,} rows...")

# make daily series
daily = pd.Series(daily_counts).sort_index()
daily.index = pd.to_datetime(daily.index)
daily = daily.asfreq("D", fill_value=0)
daily.name = "exposure_count"

daily.head(), daily.tail()


D:\Application\Anaconda\Lib\site-packages\pandas\io\sql.py:1648: SAWarning: Unrecognized server version info '17.0.1000.7'.  Some SQL Server features may not function properly.
  con = self.exit_stack.enter_context(con.connect())


Processed about 500,000 rows...
Processed about 1,000,000 rows...
Processed about 1,500,000 rows...
Processed about 2,000,000 rows...
Processed about 2,500,000 rows...
Processed about 3,000,000 rows...
Processed about 3,500,000 rows...
Processed about 4,000,000 rows...
Processed about 4,500,000 rows...
Processed about 5,000,000 rows...
Processed about 5,500,000 rows...


(2024-01-01    156
 2024-01-02    150
 2024-01-03    122
 2024-01-04    124
 2024-01-05    113
 Freq: D, Name: exposure_count, dtype: int64,
 2024-12-27    59
 2024-12-28    53
 2024-12-29    28
 2024-12-30    31
 2024-12-31    25
 Freq: D, Name: exposure_count, dtype: int64)

In [9]:
# ---------- 3. Train / test split ----------

cutoff = "2024-12-01"  # adjust if needed
train = daily.loc[:cutoff]
test  = daily.loc[cutoff:]

len(train), len(test)


(336, 31)

In [10]:
# ---------- 4. Fit and evaluate all 5 models ----------

results = {}

# 4.1 Naive: yesterday = today
naive_forecast = daily.shift(1).loc[test.index]

results["naive_mae"]  = mean_absolute_error(test, naive_forecast)
results["naive_rmse"] = np.sqrt(mean_squared_error(test, naive_forecast))


# 4.2 7-day moving average
window = 7
rolling_mean = daily.rolling(window).mean()
ma_forecast = rolling_mean.loc[test.index]

results["ma7_mae"]  = mean_absolute_error(test, ma_forecast)
results["ma7_rmse"] = np.sqrt(mean_squared_error(test, ma_forecast))


# 4.3 Simple Exponential Smoothing
ses_model = SimpleExpSmoothing(train).fit(optimized=True)
ses_forecast = ses_model.forecast(len(test))
ses_forecast.index = test.index

results["ses_mae"]  = mean_absolute_error(test, ses_forecast)
results["ses_rmse"] = np.sqrt(mean_squared_error(test, ses_forecast))


In [11]:
# 4.4 SARIMA (simple starting config with weekly seasonality)

sarima_model = SARIMAX(
    train,
    order=(1, 1, 1),             # ARIMA(p,d,q)
    seasonal_order=(1, 0, 1, 7)  # (P,D,Q,s) with s=7 days
)
sarima_fit = sarima_model.fit(disp=False)

sarima_forecast = sarima_fit.predict(start=test.index[0], end=test.index[-1])
sarima_forecast.name = "sarima"

results["sarima_mae"]  = mean_absolute_error(test, sarima_forecast)
results["sarima_rmse"] = np.sqrt(mean_squared_error(test, sarima_forecast))


In [12]:
# 4.5 Prophet

df_prophet = daily.reset_index()
df_prophet.columns = ["ds", "y"]

train_p = df_prophet[df_prophet["ds"] <= cutoff]
test_p  = df_prophet[df_prophet["ds"] >= cutoff]

m = Prophet(daily_seasonality=True, weekly_seasonality=True, yearly_seasonality=False)
m.fit(train_p)

horizon = len(test_p)
future = m.make_future_dataframe(periods=horizon, freq="D")
forecast = m.predict(future)

forecast_test = (
    forecast.set_index("ds")
            .loc[test_p["ds"]]["yhat"]
)
forecast_test.index = test.index

results["prophet_mae"]  = mean_absolute_error(test, forecast_test)
results["prophet_rmse"] = np.sqrt(mean_squared_error(test, forecast_test))


22:20:07 - cmdstanpy - INFO - Chain [1] start processing
22:20:08 - cmdstanpy - INFO - Chain [1] done processing


In [13]:
# ---------- 5. Comparison table ----------

summary = pd.DataFrame(results, index=["MAE", "RMSE"]).T
summary


,MAE,RMSE
naive_mae,11.258065,11.258065
naive_rmse,13.897087,13.897087
ma7_mae,8.317972,8.317972
ma7_rmse,9.854337,9.854337
ses_mae,9.775369,9.775369
ses_rmse,12.788376,12.788376
sarima_mae,9.822278,9.822278
sarima_rmse,12.847345,12.847345
prophet_mae,10.831354,10.831354
prophet_rmse,13.691839,13.691839


Using all five methods, the 7‑day moving average clearly produced the most accurate forecasts, with the lowest MAE and RMSE among all models. Naive, Prophet, SES, and SARIMA all improved on doing nothing, but none could beat the moving‑average baseline on December 2024 data. This means a simple, transparent 7‑day moving average is currently the best choice for forecasting daily exposure counts, and it becomes the benchmark any future model must outperform.